In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## style and bar
plt.style.use("ggplot")
color_bar = sns.color_palette()

In [3]:
path = "/content/drive/MyDrive/ml project/dataset/Dry_Bean_Dataset.xlsx"
df = pd.read_excel(path)
df.shape

(13611, 17)

In [4]:
# 1. Clean data: Remove duplicates if any exist
print(f"Initial shape: {df.shape}")
df_cleaned = df.drop_duplicates()
print(f"Shape after removing duplicates: {df_cleaned.shape}")

Initial shape: (13611, 17)
Shape after removing duplicates: (13543, 17)


In [5]:
# 2. Check for missing values
missing_values = df_cleaned.isnull().sum().sum()
print(f"Total missing values: {missing_values}")

Total missing values: 0


In [6]:
# 3. Separate features (X) and target (y)
X = df_cleaned.drop(columns=['Class'])
y = df_cleaned['Class']

In [7]:
# Encode target labels (Seker, Barbunya, Bombay, Cali, Dermosan, Horoz, Sira) into integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Encoded target classes:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


Encoded target classes: {'BARBUNYA': np.int64(0), 'BOMBAY': np.int64(1), 'CALI': np.int64(2), 'DERMASON': np.int64(3), 'HOROZ': np.int64(4), 'SEKER': np.int64(5), 'SIRA': np.int64(6)}


In [8]:
# 4. Define numerical features for scaling
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [9]:
# 5. Build the Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features)
    ]
)

In [10]:
# 6. Split data into training and testing sets (stratified to preserve class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [11]:
# 7. Fit and transform the features
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

In [12]:
print(f"Processed Training shape: {X_train_scaled.shape}")
print(f"Processed Testing shape: {X_test_scaled.shape}")

Processed Training shape: (10834, 16)
Processed Testing shape: (2709, 16)


In [13]:
# Select a single raw sample from the test partition for inference
sample_index = 0
single_raw_sample = X_test.iloc[[sample_index]]
true_label_encoded = y_test[sample_index]
true_label_name = label_encoder.inverse_transform([true_label_encoded])[0]

# Run the sample through the configured preprocessing pipeline
single_sample_scaled = preprocessor.transform(single_raw_sample)

# Print the results
print("=== Raw Input Data (Single Bean Measurements) ===")
display(single_raw_sample)

print("\n=== Preprocessed & Scaled Features (Input to ML Model) ===")
for feature, value in zip(num_features, single_sample_scaled[0]):
    print(f"{feature:15s} -> Scaled Value: {value:.6f}")

print(f"\nTrue Class: {true_label_name} (Encoded: {true_label_encoded})")
print("\nNote: Once we train our ML classifiers, we can pass 'single_sample_scaled' to model.predict() to get the prediction.")

=== Raw Input Data (Single Bean Measurements) ===


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
1223,40512,737.636,248.20203,208.525152,1.190274,0.542365,40973,227.115566,0.744268,0.988749,0.935641,0.915043,0.006127,0.00265,0.837304,0.996621



=== Preprocessed & Scaled Features (Input to ML Model) ===
Area            -> Scaled Value: -0.426017
Perimeter       -> Scaled Value: -0.545521
MajorAxisLength -> Scaled Value: -0.835117
MinorAxisLength -> Scaled Value: 0.137665
AspectRation    -> Scaled Value: -1.592981
Eccentricity    -> Scaled Value: -2.262220
ConvexArea      -> Scaled Value: -0.428269
EquivDiameter   -> Scaled Value: -0.436109
Extent          -> Scaled Value: -0.114757
Solidity        -> Scaled Value: 0.342197
roundness       -> Scaled Value: 1.044124
Compactness     -> Scaled Value: 1.865309
ShapeFactor1    -> Scaled Value: -0.386063
ShapeFactor2    -> Scaled Value: 1.559310
ShapeFactor3    -> Scaled Value: 1.955246
ShapeFactor4    -> Scaled Value: 0.356182

True Class: SEKER (Encoded: 5)

Note: Once we train our ML classifiers, we can pass 'single_sample_scaled' to model.predict() to get the prediction.
